# 02 — Train SD3.5 LoRA

Thin orchestration. Logic in `LoRA/train/`. Requires a **validated** release from notebook 01.

## 00. Clone + Git SHA

In [ ]:
import subprocess, sys
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull'], check=True)
sys.path.insert(0, str(REPO))
print('SHA', subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())

## 01. Deps + GPU preflight

In [ ]:
import os
# diffusers 0.31.0 needs transformers<=4.46 (FLAX_WEIGHTS_NAME). Force the pinned
# stack, then HARD-RESTART so Kaggle's preinstalled transformers is unloaded.
# This cell kills the kernel on purpose; run the NEXT cell after it restarts.
!pip install -q --force-reinstall --no-deps 'transformers==4.46.3' 'tokenizers==0.20.3' 'huggingface_hub==0.25.2'
!pip install -q 'diffusers==0.31.0' 'accelerate==0.34.2' 'peft==0.13.2' 'datasets>=2.20' 'bitsandbytes>=0.43' 'safetensors>=0.4.3' 'sentencepiece' 'protobuf'
print('Installed. Restarting kernel...')
os._exit(0)

In [ ]:
# Run AFTER the kernel restarts. Re-add repo to path + verify pinned stack.
import sys
sys.path.insert(0, '/kaggle/working/VIN')
import transformers, diffusers
print('transformers', transformers.__version__, '| diffusers', diffusers.__version__)
assert transformers.__version__ == '4.46.3', f'transformers={transformers.__version__}, expected 4.46.3 — re-run install + restart'
from transformers.utils import FLAX_WEIGHTS_NAME
import torch
assert torch.cuda.is_available(), 'No GPU'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1024**3,1),'GB')

## 02. Vendor the pinned trainer (see LoRA/vendor/diffusers/VENDOR.md)

In [ ]:
VEND = REPO/'LoRA'/'vendor'/'diffusers'/'v0.31.0'/'train_dreambooth_lora_sd3.py'
if not VEND.exists():
    VEND.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['wget','-q','https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/dreambooth/train_dreambooth_lora_sd3.py','-O',str(VEND)], check=True)
print('trainer:', VEND, VEND.exists())

## 03. Verify validated release

In [ ]:
from LoRA.data.validate import require_validated_release
WORK = Path('/kaggle/working/vin_lora')
RELEASE_DIR = WORK/'releases'/'pedestrian_lora_v1'
print(require_validated_release(RELEASE_DIR))

## 04. Dry run

In [ ]:
from LoRA.train.train_sd35_lora import run_training
run_training(RELEASE_DIR, WORK, dry_run=True)

## 05. Smoke train (100 steps)

In [ ]:
smoke = run_training(RELEASE_DIR, WORK, dry_run=False, max_train_steps=100)
print(smoke['run_dir'], smoke['verification'])

## 06. First experiment (1000 steps)

In [ ]:
train = run_training(RELEASE_DIR, WORK, dry_run=False)  # uses lora_train.yaml steps
ADAPTER = train['adapter_path']
print('adapter:', ADAPTER)
print('verify :', train['verification'])

## 07. Zip artifacts

In [ ]:
import shutil
run_dir = train['run_dir']
zip_path = shutil.make_archive(str(run_dir), 'zip', str(run_dir))
print('artifact zip:', zip_path)